# 🧬 ⚙️ CpGPT Attention Tutorial ⚙️ 🧬

Welcome to the CpGPT Attention Tutorial! 👋 

In this notebook, we'll walk you through the process of retrieving and interpreting attention weights for a pre-trained CpGPT model.

## Table of Contents

1. [Setup Environment](#1-setup-environment)
2. [Load CpGPT Dependencies](#2-load-cpgpt-dependencies)
3. [Load Pre-trained Model](#3-load-pre-trained-model)
4. [Save Data of Interest](#4-save-data-of-interest)
5. [Generate DNA LLM Embeddings](#5-generate-dna-llm-embeddings)
6. [Create Sorted and Original Datasets](#6-create-sorted-and-original-datasets)
7. [Retrieve Attention Weights](#7-retrieve-attention-weights)
8. [Visualize Attention Weights](#8-visualize-attention-weights)
9. [Get Most Important CpGs](#9-get-most-important-cpgs)
10. [Next Steps](#10-next-steps)

## 1. Setup Environment

We'll import the necessary Python packages and set up our environment for fine-tuning CpGPT. We'll be using a mix of standard data science libraries and CpGPT-specific modules. We'll also set some important variables that will be used throughout the notebook. Pay attention to these as you may need to adjust them based on your specific setup and requirements. The bulk of the variables are defined here below. The pre-trained model checkpoint alongside DNA embedding files can also be downloaded if not present.

CpGPT model files and DNA-sequence dependencies are hosted on Hugging Face.
The download cell below downloads any missing files into the standard Hugging Face cache and reuses them on later runs; no command-line download is needed.

### 1.1 Download dependencies and define variables

In [ ]:
from cpgpt import download_cpgpt

resources = download_cpgpt(model="small", species="human")

In [ ]:
# Random seed for reproducibility
RANDOM_SEED = 42

# Directory paths
PROCESSED_DIR = "../data/tutorials/processed/attend"
ARROW_DF_PATH = "../data/tutorials/raw/toy.arrow"

# The maximum context length to give to the model (above 5k is difficult to make a heatmap and above 10k you might run out of GPU memory)
MAX_INPUT_LENGTH = 2_000

# Device configuration
DEVICE = "cuda"

LLM_DEPENDENCIES_DIR = str(resources.dependencies_path)
DEPENDENCIES_DIR = str(resources.dependencies_path)
MODEL_CHECKPOINT_PATH = str(resources.checkpoint_path)
MODEL_CONFIG_PATH = str(resources.config_path)
MODEL_VOCAB_PATH = str(resources.vocab_path) if resources.vocab_path is not None else None

### 1.2 Import packages


In [ ]:
# Standard library imports
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

# Plotting imports
import matplotlib.pyplot as plt

plt.style.use(
    [
        "nature",
    ],
)
import numpy as np
import pandas as pd
import seaborn as sns
import torch

# Lightning imports
from lightning.pytorch import seed_everything
from matplotlib.patches import Rectangle

# Rich imports
from rich.console import Console
from rich.table import Table

# cpgpt-specific imports
from cpgpt.data.components.cpgpt_datasaver import CpGPTDataSaver
from cpgpt.data.components.cpgpt_dataset import CpGPTDataset
from cpgpt.data.components.dna_llm_embedder import DNALLMEmbedder
from cpgpt.data.components.illumina_methylation_prober import IlluminaMethylationProber
from cpgpt.infer.cpgpt_inferencer import CpGPTInferencer
from cpgpt.utils.rich_utils import create_rich_dataset_preview

# Set random seed for reproducibility
seed_everything(RANDOM_SEED, workers=True)

## 2. Load CpGPT Dependencies

Now that we have our basic environment set up, we need to load three crucial components for CpGPT:

1. **DNA LLM Embedder**: This component contains the embeddings for all CpGs of interest. It's essential for converting DNA sequences into a format that our model can understand and process.

2. **Illumina Methylation Prober**: This tool allows us to convert probe names to genomic locations, which is required for working with Illumina methylation array data.

3. **CpG Inferencer**: This component is responsible for loading the pre-trained CpGPT model checkpoint and any additional configurations or overrides needed for the fine-tuning process.

### 3.1 Load DNALLMEmbedder

In [ ]:
embedder = DNALLMEmbedder(dependencies_dir=DEPENDENCIES_DIR)

### 3.2 Load IlluminaMethylationProber

In [ ]:
prober = IlluminaMethylationProber(dependencies_dir=DEPENDENCIES_DIR, embedder=embedder)

### 3.3 Load CpGInferencer

In [ ]:
inferencer = CpGPTInferencer()

## 3. Load Pre-trained Model

We need to load a pre-trained CpGPT model to be able to reconstruct the beta values for different CpG sites. If you would like to get the attention weights for what seems important for the methylome in general, use the main pre-trained CpGPT model. However, if you'd like to see how the model attend to CpG sites to predict a certain condition, then use a finetuned model.

In [ ]:
config = inferencer.load_cpgpt_config(MODEL_CONFIG_PATH)
model_info = {"current_hparams": config}
model = inferencer.load_cpgpt_model(
    config,
    model_ckpt_path=MODEL_CHECKPOINT_PATH,
    strict_load=True,
)

## 4. Save Data of Interest

We'll prepare our methylation data to retrieve attention weights and save it in a format that CpGPT can efficiently process. We first begin with a dataset that is already stored in .arrow (feather v2) format. Then, let's create a memory-mapped dataset.

In [ ]:
# Read the arrow file
df = pd.read_feather(ARROW_DF_PATH)

# Show first few rows
df.head()

In [ ]:
# Define datasaver
attention_datasaver = CpGPTDataSaver(data_paths=ARROW_DF_PATH, processed_dir=PROCESSED_DIR)

# Process the file
attention_datasaver.process_files(prober, embedder)

## 5. Generate DNA LLM Embeddings
 
In this section, we'll generate any DNA embeddings for the genomic locations of our dataset that might be missing. The model requires that all genomic locations have embeddings from a DNA LLM or it will fail otherwise. The easiest way is to download such embeddings if you are working with human samples ([step 1.3](#1.3-download-dependencies)) and Illumina arrays or it might take a little bit for the processing to happen.

In [ ]:
# Get all unique species from the datasavers
all_species = list(attention_datasaver.all_genomic_locations.keys())

# Loop through each species
for species in all_species:
    all_genomic_locations = list(attention_datasaver.all_genomic_locations.get(species, []))

    embedder.parse_dna_embeddings(
        all_genomic_locations,
        species,
        dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
        dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    )

## 6. Create Sorted and Original Datasets

We need to define a way that the CpGPT model can efficiently access our memory-mapped dataset. To facilitate the visualization of the attention weights, we'll create two datasets:

1. **Sorted Dataset**: This dataset is sorted by the genomic locations of the CpG sites, with chromosomes shuffled. This will allow us to see how the model attends to different regions of the genome.
2. **Original Dataset**: This dataset is in the same order as the original data. This is convenient for tracing back the attention weights to the original features.

Let's define the CpGPTDataset objects and check the outputs.

In [ ]:
sorted_data = CpGPTDataset(
    embedder,
    processed_dir=PROCESSED_DIR,
    max_length=MAX_INPUT_LENGTH,
    sorting_strategy="sorted_chromosome",
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
)

unsorted_data = CpGPTDataset(
    embedder,
    processed_dir=PROCESSED_DIR,
    max_length=MAX_INPUT_LENGTH,
    sorting_strategy="original",
    dna_context_len=model_info["current_hparams"]["data"]["dna_context_len"],
    dna_llm=model_info["current_hparams"]["data"]["dna_llm"],
)

Let's double check the outputs make sense:

In [ ]:
create_rich_dataset_preview(sorted_data, "Sorted Data Sample")

create_rich_dataset_preview(unsorted_data, "Unsorted Data Sample")

## 7. Retrieve Attention Weights

Let's get the attention weights for the sorted and unsorted datasets for a few samples. Typically, the last attention layer is chosen and the average accross all heads is taken but that can also be changed.

In [ ]:
# Define samples
n_samples = 3
sorted_samples = [sorted_data[i] for i in range(n_samples)]
unsorted_samples = [unsorted_data[i] for i in range(n_samples)]

# Get attention weights
sorted_attn_weights = [
    inferencer.get_attention_weights(model, sample, layer_index=-1, aggregate_heads="mean")
    .detach()
    .cpu()
    .numpy()
    for sample in sorted_samples
]
unsorted_attn_weights = [
    inferencer.get_attention_weights(model, sample, layer_index=-1, aggregate_heads="mean")
    .detach()
    .cpu()
    .numpy()
    for sample in unsorted_samples
]

## 8. Visualize Attention Weights
 
In this section, we'll visualize the attention weights for the sorted datasets as heatmaps. Let's see if the model learned anything about the 3D genomic structure.

In [ ]:
# Heatmap of sorted data for three samples

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=150)
fig.suptitle("Mean Attention Weights of Last Layer with Chromosome Boundaries", fontsize=16)


def normalize_attention(attn_weights, normalize=True, dim=0):
    attn_weights_no_cls = attn_weights[1:, 1:]
    if normalize:
        return torch.nn.functional.softmax(torch.tensor(attn_weights_no_cls), dim=dim).numpy()
    return attn_weights_no_cls


normalize = False  # Set this to False if you don't want to normalize

for i, (attn_weights, sample) in enumerate(
    zip(sorted_attn_weights[:3], sorted_samples[:3], strict=False),
):
    ax = axes[i]

    # Normalize the attention weights if specified
    processed_attn_weights = normalize_attention(attn_weights, normalize=normalize)

    # Create the heatmap with processed weights
    sns.heatmap(processed_attn_weights, cmap="viridis", robust=True, ax=ax)

    # Get the chromosome labels
    chromosome_labels = sample[2].cpu().numpy()

    # Find the unique chromosomes and their start/end indices
    unique_chromosomes = np.unique(chromosome_labels)
    for chrom in unique_chromosomes:
        chrom_positions = np.where(chromosome_labels == chrom)[0]
        start_idx = chrom_positions[0]
        end_idx = chrom_positions[-1] + 1  # +1 because the end index is exclusive

        rect = Rectangle(
            (start_idx, start_idx),
            end_idx - start_idx,
            end_idx - start_idx,
            fill=False,
            edgecolor="red",
            linewidth=1,
        )
        ax.add_patch(rect)

    ax.set_title(f"Sample {i}")

plt.tight_layout()
plt.show()

## 9. Get Most Important CpGs

Using the unsorted data, we can get the most important CpGs for a given sample. However, keep in mind that if the dataset has more features than the maximum context length set at the very beginning of this tutorial, then only the first N features will be retrieved.

In [ ]:
# Define helper variables
reverse_dict = {v: k for k, v in embedder.ensembl_metadata_dict["homo_sapiens"]["vocab"].items()}
k = 10

console = Console()

# Function to create and print table


def create_and_print_table(data, title) -> None:
    table = Table(title=title)
    for column in data.columns:
        table.add_column(
            column,
            style=(
                "cyan"
                if column == "Probe"
                else (
                    "magenta"
                    if column == "Genomic Location"
                    else "yellow"
                    if column == "Std Attention Score"
                    else "green"
                )
            ),
        )
    for _, row in data.iterrows():
        table.add_row(*[f"{val:.6f}" if isinstance(val, float) else str(val) for val in row])
    console.print(table)

In [ ]:
# Show top CpGs for individual samples
for i, sample in enumerate(unsorted_samples[:3]):
    cls_attention = unsorted_attn_weights[i][0, 1:]
    chromosomes = sample[2].cpu().numpy()
    positions = sample[3].cpu().numpy()

    genomic_locations = [
        f"{reverse_dict[chrom]}:{pos}" for chrom, pos in zip(chromosomes, positions, strict=False)
    ]
    probes = prober.probe_location(genomic_locations, species="homo_sapiens")

    sample_results = pd.DataFrame(
        {"Probe": probes, "Genomic Location": genomic_locations, "Attention Score": cls_attention},
    )

    sample_results = sample_results.sort_values("Attention Score", ascending=False).reset_index(
        drop=True,
    )
    top_k_sample = sample_results.head(k)

    create_and_print_table(
        top_k_sample,
        f"Top {k} CpG sites with highest attention from CLS token (Sample {i})",
    )
    console.print("\n")

In [ ]:
# Initialize a dictionary to store attention scores for each CpG site across all samples
attention_scores = {}

# Loop through all samples
for i, sample in enumerate(unsorted_samples):
    cls_attention = unsorted_attn_weights[i][0, 1:]
    chromosomes = sample[2].cpu().numpy()
    positions = sample[3].cpu().numpy()

    genomic_locations = [
        f"{reverse_dict[chrom]}:{pos}" for chrom, pos in zip(chromosomes, positions, strict=False)
    ]
    probes = prober.probe_location(genomic_locations, species="homo_sapiens")

    # Update the attention_scores dictionary
    for probe, location, score in zip(probes, genomic_locations, cls_attention, strict=False):
        if probe not in attention_scores:
            attention_scores[probe] = {"location": location, "scores": []}
        attention_scores[probe]["scores"].append(score)

# Calculate mean and std of attention scores
for probe in attention_scores:
    scores = attention_scores[probe]["scores"]
    attention_scores[probe]["mean"] = np.mean(scores)
    attention_scores[probe]["std"] = np.std(scores)

# Create a DataFrame with the results
results_df = pd.DataFrame(
    [
        {
            "Probe": probe,
            "Genomic Location": data["location"],
            "Mean Attention Score": data["mean"],
            "Std Attention Score": data["std"],
        }
        for probe, data in attention_scores.items()
    ],
)

# Sort the DataFrame by Mean Attention Score in descending order
results_df = results_df.sort_values("Mean Attention Score", ascending=False).reset_index(drop=True)

# Get top k results
top_k_results = results_df.head(k)

create_and_print_table(
    top_k_results,
    f"Top {k} CpG sites with highest mean attention from CLS token across all samples",
)

## 10. Next steps

Here are some ideas for further exploration:

- Experiment with different input context lengths
- Look at the attention weights accross different layers
- See how the attention differs across finetuned models
- Use max aggregation for the attention heads
- Calculate gene and chromatin enrichment for the top CpG sites